# LAB 09: Subword Tokenization & Label Alignment
**Học phần:** Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Chủ đề:** Xử lý ranh giới từ và căn chỉnh nhãn (Label Alignment) trong Token Classification với Pretrained Transformer


## 1. Vấn Đề Trung Tâm: Subword Tokenization & Label Alignment
Khi chuyển từ cấp độ từ (word-level) sang Transformer token IDs (subword-level), một từ trong câu có thể bị tách thành nhiều mẩu subword (ví dụ: `playing` $\rightarrow$ `play` + `##ing`).  
Do đó, nhãn thực thể gốc của từ cần được căn chỉnh (align) tương ứng sang chuỗi subword tokens.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")

sample_words = ["Barack", "Obama", "visited", "Washington", "playing", "football", "."]
encoding = tokenizer(sample_words, is_split_into_words=True)
subwords = tokenizer.convert_ids_to_tokens(encoding["input_ids"])

print(f"Từ gốc: {sample_words}")
print(f"Subwords sinh ra: {subwords}")
print(f"Word IDs: {encoding.word_ids()}")


## 2. Chiến Lược Căn Chỉnh Nhãn (Alignment Strategies) & Ignore Index

### Lựa chọn chiến lược: Strategy A
- **Mô tả:** Gán nhãn thực thể gốc cho token subword đầu tiên của từ. Tất cả các token subword tiếp theo (`##...`) và các token đặc biệt (`[CLS]`, `[SEP]`, padding) được gán nhãn `IGNORE` (quy ước giá trị `-100`).
- **Giải thích lý do lựa chọn:**
  1. *Tương thích PyTorch CrossEntropyLoss:* Trong PyTorch, tham số `ignore_index=-100` giúp tự động loại bỏ các vị trí này khỏi việc tính toán hàm mất mát:
     $$L = - \sum_{i: y_i \neq -100} \log P(y_i | x)$$
  2. *Tránh phạt kép (Over-penalization):* Nếu gán nhãn cho tất cả các subwords (Strategy B), một từ dài bị chia nhiều mẩu sẽ đóng góp quá nhiều gradient vào loss so với từ ngắn 1 subword, làm mô hình thiên lệch.
  3. *Bảo toàn ranh giới BIO:* Token subword đầu tiên là nơi chứa thông tin ranh giới bắt đầu (`B-X`).
- **Ý nghĩa của IGNORE:** Token mang nhãn `IGNORE` vẫn tham gia tính toán self-attention để cung cấp ngữ cảnh đầy đủ, nhưng gradient của nhãn dự đoán tại vị trí đó không được cộng dồn vào hàm mất mát.


## 3. Core Implementation: Căn Chỉnh Nhãn & Kiểm Tra Tensor
Triển khai hàm căn chỉnh nhãn nhất quán theo Strategy A và quan sát đầy đủ các thành phần: `tokens`, `input_ids`, `attention_mask`, `word_ids`, `labels`, `logits`, `predictions`.


In [ ]:
import torch
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
model.eval()

words = ["Elon", "Musk", "founded", "SpaceX", "in", "California", "."]
# Giả sử nhãn word-level theo id của CoNLL (3: B-PER, 4: I-PER, 0: O, 5: B-ORG, 7: B-LOC)
word_labels = [3, 4, 0, 5, 0, 7, 0]

# Tokenize với is_split_into_words=True
inputs = tokenizer(
    words,
    is_split_into_words=True,
    truncation=True,
    padding=True,
    return_tensors="pt"
)

word_ids = inputs.word_ids(batch_index=0)

# Xây dựng token_labels theo Strategy A
token_labels = []
prev_word_idx = None
for word_idx in word_ids:
    if word_idx is None:
        token_labels.append(-100) # Special tokens [CLS], [SEP]
    elif word_idx != prev_word_idx:
        token_labels.append(word_labels[word_idx])
    else:
        token_labels.append(-100) # Subword tiếp theo gán IGNORE (-100)
    prev_word_idx = word_idx

# Suy luận forward pass qua mô hình
with torch.inference_mode():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)[0]

# Hiển thị cấu trúc pipeline chi tiết
tokens_list = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"{'Token':<12} | {'Input ID':<8} | {'Mask':<4} | {'Word ID':<8} | {'Label ID':<8} | {'Pred ID':<8} | {'Pred Tag':<8}")
print("-" * 75)
for t, iid, m, wid, lid, pid in zip(
    tokens_list,
    inputs["input_ids"][0].tolist(),
    inputs["attention_mask"][0].tolist(),
    word_ids,
    token_labels,
    predictions.tolist(),
):
    wid_str = str(wid) if wid is not None else "None"
    pred_tag = model.config.id2label[pid]
    print(f"{t:<12} | {iid:<8} | {m:<4} | {wid_str:<8} | {lid:<8} | {pid:<8} | {pred_tag:<8}")


## 4. Experiment 1: Tokenization Analysis
Chọn 20 câu trong tập CoNLL-2003, thống kê số lượng words, tokens, số lần subword split và tỷ lệ $R = \frac{\#\text{subword tokens}}{\#\text{words}}$.


In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset('lhoestq/conll2003')
sample_20 = ds['train'][:20]

stats = []
for i in range(20):
    words_i = sample_20['tokens'][i]
    enc = tokenizer(words_i, is_split_into_words=True)
    raw_tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'])
    # Bỏ qua [CLS] và [SEP] để so sánh số lượng token với từ gốc
    subword_tokens = [t for t in raw_tokens if t not in ['[CLS]', '[SEP]']]
    
    n_words = len(words_i)
    n_tokens = len(subword_tokens)
    splits = n_tokens - n_words
    r = n_tokens / n_words if n_words > 0 else 1.0
    
    stats.append({
        "Sentence": i,
        "Words": n_words,
        "Tokens": n_tokens,
        "Subword Splits": splits,
        "Ratio R": round(r, 4),
        "Snippet": " ".join(words_i)[:40] + "..."
    })

df_stats = pd.DataFrame(stats)
print(df_stats.to_string(index=False))
print(f"\nTrung bình tỷ lệ R trên 20 câu: {df_stats['Ratio R'].mean():.4f}")


In [ ]:
# So sánh phân nhóm: Câu ngắn vs Câu dài, Entity vs Non-Entity
median_len = df_stats['Words'].median()
short_sents = df_stats[df_stats['Words'] <= median_len]
long_sents = df_stats[df_stats['Words'] > median_len]

print(f"Tỷ lệ R trung bình ở câu ngắn (<= {median_len:.0f} words): {short_sents['Ratio R'].mean():.4f}")
print(f"Tỷ lệ R trung bình ở câu dài (> {median_len:.0f} words): {long_sents['Ratio R'].mean():.4f}")

# So sánh Entity tokens vs Non-Entity tokens
entity_splits = 0
entity_words = 0
non_entity_splits = 0
non_entity_words = 0

for i in range(20):
    words_i = sample_20['tokens'][i]
    tags_i = sample_20['ner_tags'][i]
    for w, t in zip(words_i, tags_i):
        w_enc = tokenizer.tokenize(w)
        if t != 0: # Entity
            entity_words += 1
            entity_splits += (len(w_enc) - 1)
        else: # Non-entity
            non_entity_words += 1
            non_entity_splits += (len(w_enc) - 1)

print(f"\nEntity tokens: {entity_words} từ, số mẩu tách: {entity_splits} (Tỷ lệ phân rã: {entity_splits/entity_words:.4f})")
print(f"Non-entity tokens: {non_entity_words} từ, số mẩu tách: {non_entity_splits} (Tỷ lệ phân rã: {non_entity_splits/non_entity_words:.4f})")


### Giải thích yêu cầu: Những loại từ nào thường bị chia thành nhiều subword?
1. **Tên riêng hiếm gặp (Rare Proper Nouns):** Các tên người, địa danh thuộc các ngôn ngữ hoặc quốc gia ít phổ biến (ví dụ: *Fischler*, *Loyola*, *Cuttitta*) không nằm trong từ vựng gốc của WordPiece nên bị băm nhỏ thành các ký tự hoặc âm tiết.
2. **Từ ghép có dấu gạch ngang (Hyphenated & Compound Words):** Các từ kết hợp như *EU-wide*, *fast-food*, *1996-08-22* thường bị chia tách ở dấu gạch nối và các thành phần liên quan.
3. **Từ phái sinh và hình thái ngữ pháp phức tạp (Morphological Inflections):** Các từ có tiền tố/hậu tố ít phổ biến hoặc dạng thức ngữ pháp đặc biệt (ví dụ: *multidisciplinary* $\rightarrow$ `multi` + `##disciplinary`).
4. **Số liệu, mã hiệu, ngày tháng:** Các chuỗi số hoặc ký hiệu kỹ thuật thường bị tách thành từng cụm chữ số.
